# B2-020 — Session 2: Causal Transformer Language Model

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).



## 1. Language Transformer input contract

A language-transformer maps integer token IDs `(B,N)` to token embeddings `(B,N,8)`, adds B2-019's sinusoidal positional table, runs one pre-norm Transformer block, and projects each position to 12 vocabulary logits `(B,N,12)`.

**Checkpoint 1A.** Trace all four shapes.

**Checkpoint 1B.** Which axis indexes vocabulary logits?

In [ ]:
import torch
torch.manual_seed(20260812)
ATOL = 1e-6
RTOL = 1e-6
assert torch.get_num_threads() >= 1

## 2. Reuse the causal mask

B2-019 established attention masks and causal-self-attention. Here an upper-triangular forbidden region is applied before softmax, so row `i` may attend only to key positions `j <= i`; padding is also excluded.

**Checkpoint 2A.** Write the allowed condition on `i,j`.

**Checkpoint 2B.** Why is masking after softmax wrong?

## 3. Shift-right labels

For token row `[t0,t1,...,t7]`, inputs are `tokens[:,:-1]` and targets are `tokens[:,1:]`. Logit position `i` predicts the next token and never receives that target as an input at the same position.

**Checkpoint 3A.** Give both shapes for batch 3.

**Checkpoint 3B.** Identify the off-by-one leakage mutation.

## 4. Token loss and padding

Flatten `(B,N-1,12)` logits and `(B,N-1)` targets only after aligning them. Mean cross-entropy includes exactly target positions whose IDs are non-padding; padding contributes neither numerator nor denominator.

**Checkpoint 4A.** Write the valid-token mask.

**Checkpoint 4B.** Why can averaging per row change the answer?

## 5. Worked tiny causal LM trace

With seed `20260812`, follow token embedding plus sinusoidal position, causal attention, residual/LayerNorm/feed-forward, vocabulary head, shifted loss, backward, and AdamW step. The vocabulary head is distinct and not weight-tied to the embedding table.

**Checkpoint 5A.** Which parameter consumes width 8 and emits 12?

**Checkpoint 5B.** Where does the causal mask enter?

## 6. Training and held-out evaluation

Train at most 80 fixed updates on literal rows and evaluate held-out next-token loss without updating parameters. A training loss decrease is not a held-out certificate; report both and keep split IDs disjoint.

**Checkpoint 6A.** When must `eval()` be used?

**Checkpoint 6B.** What does dropout 0.0 simplify?

## 7. Common pitfalls

Common failures are unshifted targets, a diagonal-forbidden mask, broadcasting padding across the wrong axis, and returning logits as `(B,12,N)`. Repair by pinning shape probes and checking one position manually.

**Checkpoint 7A.** Which error hides the current token?

**Checkpoint 7B.** Which error leaks the next token?

## 8. Exam connections and Going deeper

Expect shape reconstruction, causal-dependence proofs, target repair, and held-out loss comparisons. Going deeper names cached decoding and subword vocabularies, but required work stays with explicit token IDs and no tokenizer library.

**Checkpoint 8A.** Which practice proves nondependence?

**Checkpoint 8B.** Why is cached decoding not needed for training?